# WM-811K: supervised vs semi-supervised sweep (Colab A100)

Runs the full label-budget sweep and writes the results chart. Select **Runtime > Change runtime type > A100 GPU** first.

Before the first run:
1. Locally, run `python -m wm811k.cli prepare` (needs `LSWMD.pkl`). It writes `artifacts/experiment_data.npz`, a small compressed cache of the resized maps and splits.
2. Upload that file to Google Drive at `MyDrive/wm811k/experiment_data.npz`.
3. Push this repo to GitHub and set `REPO_URL` below.

Everything is resumable: finished runs are skipped, and the MLflow database and SimCLR checkpoint are copied to Drive after each arm.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = ""  # e.g. "https://github.com/<you>/<repo>.git"
assert REPO_URL, "Set REPO_URL to the GitHub URL of this project"

DRIVE_DIR = "/content/drive/MyDrive/wm811k"
WORK_DIR = "/content/work"

In [ ]:
!git clone -q $REPO_URL /content/wm811k
!pip install -q hydra-core mlflow xgboost scikit-learn pandas matplotlib
!mkdir -p $WORK_DIR
# restore anything saved by an earlier session: cache, MLflow database, SimCLR checkpoint
!cp -n $DRIVE_DIR/experiment_data.npz $WORK_DIR/ 2>/dev/null; cp -n $DRIVE_DIR/mlflow.db $WORK_DIR/ 2>/dev/null; cp -n $DRIVE_DIR/simclr_encoder.pt $WORK_DIR/ 2>/dev/null; ls -la $WORK_DIR

In [ ]:
import os
import subprocess

OVERRIDES = [
    f"data.cache={WORK_DIR}/experiment_data.npz",
    f"mlflow.tracking_uri=sqlite:///{WORK_DIR}/mlflow.db",
    f"simclr.checkpoint={WORK_DIR}/simclr_encoder.pt",
    f"artifacts_dir={WORK_DIR}/artifacts",
    "train.num_workers=4",
]


def wm811k(command, *args):
    """Run `python -m wm811k.cli <command> ...` from the repo and stream its output."""
    env = {**os.environ, "PYTHONPATH": "/content/wm811k/src", "MLFLOW_DISABLE_AGENT_HINT": "1"}
    subprocess.run(
        ["python", "-m", "wm811k.cli", command, *args, *OVERRIDES],
        cwd="/content/wm811k", env=env, check=True,
    )


def save_progress():
    !cp $WORK_DIR/mlflow.db $DRIVE_DIR/ 2>/dev/null; cp $WORK_DIR/simclr_encoder.pt $DRIVE_DIR/ 2>/dev/null; echo saved

## Sweep
One arm per cell so progress is saved between arms. About 15 runs per arm (5 budgets x 3 seeds).

In [ ]:
wm811k('sweep', 'gbm')
save_progress()

In [ ]:
wm811k('sweep', 'cnn')
save_progress()

In [ ]:
wm811k('sweep', 'simclr')  # pretrains once (steps=simclr.steps), then fine-tunes
save_progress()

In [ ]:
wm811k('sweep', 'fixmatch')
save_progress()

## Results

In [ ]:
wm811k("results")
!cp -r $WORK_DIR/artifacts $DRIVE_DIR/
from IPython.display import Image

Image(f"{WORK_DIR}/artifacts/label_efficiency.png")

Download `mlflow.db` and the `artifacts/` folder from Drive into the repo root. `mlflow.db` holds every run, so the best model can be exported for serving locally (see the README).